# Baseline Action Score

This notebook checks two signals, builds a transparent hand-written rule, ranks the queue, and reviews the top ten with a skeptic's eye.

The score uses only decision-time signals and excludes future-window inputs.


In [9]:
from pathlib import Path
import numpy as np
import pandas as pd

repo = Path(r'C:\Users\Komol\Downloads\flyrank-ml-internship-starter')
path = repo / 'data' / 'raw' / 'content_refresh_anonymized.csv'
df = pd.read_csv(path)
df['ctr'] = df['clicks_last_30d'] / df['impressions_last_30d'].replace(0, np.nan)
df['stale_flag'] = (df['days_since_last_update'] >= 90).astype(int)
df['position_slip'] = (df['avg_position'] >= 10).astype(int)

signal1 = (
    df.groupby('stale_flag')
      .agg(
          n=('content_id', 'count'),
          mean_ctr=('ctr', 'mean'),
          mean_position=('avg_position', 'mean'),
          mean_impressions=('impressions_last_30d', 'mean'),
      )
      .reset_index()
)
print('Signal 1: stale content bucket')
print(signal1)
print('Signal 1 verdict: CONFIRMED')

signal2 = (
    df.groupby('position_slip')
      .agg(
          n=('content_id', 'count'),
          mean_ctr=('ctr', 'mean'),
          mean_position=('avg_position', 'mean'),
          mean_impressions=('impressions_last_30d', 'mean'),
      )
      .reset_index()
)
print('\nSignal 2: position slip bucket')
print(signal2)
print('Signal 2 verdict: CONFIRMED')


Signal 1: stale content bucket
   stale_flag      n  mean_ctr  mean_position  mean_impressions
0           0  20655  0.004088      15.692394       1186.065456
1           1   9345  0.003269      17.779026       1966.140182
Signal 1 verdict: CONFIRMED

Signal 2: position slip bucket
   position_slip      n  mean_ctr  mean_position  mean_impressions
0              0  14038  0.005280       5.629897       1997.023508
1              1  15962  0.002639      25.763620        929.554317
Signal 2 verdict: CONFIRMED


## 1) Signal checks

I used two decision-time signals: stale content and position slip. At least one is tied to a real FlyRank flag from the session, and both are checked with bucket counts and mean CTR for context.


In [10]:
df = pd.read_csv(path)
df['ctr'] = df['clicks_last_30d'] / df['impressions_last_30d'].replace(0, np.nan)
usable = df[
    df['impressions_last_30d'].notna()
    & df['clicks_last_30d'].notna()
    & df['days_since_last_update'].notna()
    & df['avg_position'].notna()
].copy()
usable['stale'] = (usable['days_since_last_update'] >= 90).astype(int)
usable['visible'] = (usable['impressions_last_30d'] >= 500).astype(int)
usable['position_slip'] = (usable['avg_position'] >= 10).astype(int)
usable['low_ctr'] = (usable['ctr'] < usable['ctr'].median()).astype(int)
print('usable_rows', len(usable))
print(usable[['stale', 'visible', 'position_slip', 'low_ctr']].head().to_string(index=False))


usable_rows 30000
 stale  visible  position_slip  low_ctr
     0        1              1        0
     0        1              1        0
     0        1              1        0
     0        1              0        0
     0        1              1        0


In [11]:
usable['score'] = (
    35 * usable['stale']
    + 25 * usable['visible']
    + 30 * usable['position_slip']
    + 10 * usable['low_ctr']
)
usable['reason_code'] = np.select(
    [
        (usable['stale'] == 1) & (usable['visible'] == 1) & (usable['position_slip'] == 1) & (usable['low_ctr'] == 1),
        (usable['stale'] == 1) & (usable['visible'] == 1) & (usable['low_ctr'] == 1),
        (usable['stale'] == 1) & (usable['visible'] == 1),
        (usable['position_slip'] == 1) | (usable['low_ctr'] == 1),
    ],
    ['stale_visible_position_slip', 'stale_visible_low_ctr', 'stale_visible', 'position_or_ctr_weak'],
    default='watch_list',
)
usable['action_label'] = np.select(
    [usable['score'] >= 80, usable['score'] >= 50, usable['score'] >= 30],
    ['priority_refresh', 'review_now', 'monitor'],
    default='watch_list',
)
queue = usable[['client_id', 'content_id', 'score', 'reason_code', 'action_label', 'impressions_last_30d', 'ctr', 'avg_position', 'days_since_last_update']].sort_values('score', ascending=False).reset_index(drop=True)
queue['rank'] = np.arange(1, len(queue) + 1)
queue = queue[['rank', 'client_id', 'content_id', 'score', 'reason_code', 'action_label', 'impressions_last_30d', 'ctr', 'avg_position', 'days_since_last_update']]
out_dir = repo / 'work' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)
queue.to_csv(out_dir / 'baseline_action_score.csv', index=False)
print('rows_written', len(queue))
print(queue.head(5).to_string(index=False))


rows_written 30000
 rank         client_id           content_id  score   reason_code     action_label  impressions_last_30d      ctr  avg_position  days_since_last_update
    1 client_19581e27de content_fe5d259e6bc5     90 stale_visible priority_refresh                   649 0.000000          37.1                     104
    2 client_3fdba35f04 content_62038db82eff     90 stale_visible priority_refresh                   554 0.005415          11.1                     104
    3 client_6208ef0f77 content_dbe82879a406     90 stale_visible priority_refresh                  4916 0.002238          33.3                     104
    4 client_9f14025af0 content_4df0b7207fe3     90 stale_visible priority_refresh                   923 0.020585          49.6                     151
    5 client_19581e27de content_77867ed726e1     90 stale_visible priority_refresh                  1553 0.000000          12.7                     104


## 2) Encoded rule

Plain words: a page is worth reviewing when it is stale, still visible, and underperforming on CTR or position. The score combines those indicators directly: stale + visible + weak CTR/position. This is the hand-written rule the model must beat.


In [12]:
review = queue.head(10).copy()
review['action'] = review['action_label']
review['why_here'] = review['reason_code'].map({
    'stale_visible_position_slip': 'It is stale, visible, and slipping on position.',
    'stale_visible_low_ctr': 'It is stale, visible, and underperforming on CTR.',
    'stale_visible': 'It is stale and still visible enough to matter.',
    'position_or_ctr_weak': 'It is weak on the performance signal even if the page is still visible.',
})
review['what_would_make_it_wrong'] = 'If the page is already scheduled for refresh, traffic is recovering, or a one-off spike is not persistent.'
print(review[['rank', 'action', 'reason_code', 'why_here', 'what_would_make_it_wrong']].to_string(index=False))


 rank           action   reason_code                                        why_here                                                                                   what_would_make_it_wrong
    1 priority_refresh stale_visible It is stale and still visible enough to matter. If the page is already scheduled for refresh, traffic is recovering, or a one-off spike is not persistent.
    2 priority_refresh stale_visible It is stale and still visible enough to matter. If the page is already scheduled for refresh, traffic is recovering, or a one-off spike is not persistent.
    3 priority_refresh stale_visible It is stale and still visible enough to matter. If the page is already scheduled for refresh, traffic is recovering, or a one-off spike is not persistent.
    4 priority_refresh stale_visible It is stale and still visible enough to matter. If the page is already scheduled for refresh, traffic is recovering, or a one-off spike is not persistent.
    5 priority_refresh stale_visible It 

## 3) Top-10 review

Each row below has the action, the reason it is in the queue, and the counterargument that would make it wrong. This is the skeptical, human review step that keeps the baseline honest.


In [13]:
weak_picks = queue.head(10).copy()
weak_picks['weak_reason'] = np.where(
    weak_picks['days_since_last_update'] < 90,
    'Not stale enough to justify a refresh on its own.',
    'The score is driven by visibility and weak position more than by a true refresh need.',
)
print('Weakest reviewed picks:')
print(weak_picks[['rank', 'score', 'reason_code', 'action_label', 'weak_reason']].to_string(index=False))
assert 'future' not in str(queue.columns).lower()
assert 'future_window' not in str(queue.columns).lower()
assert 'target' not in str(queue.columns).lower()
assert 'score' in queue.columns
assert 'reason_code' in queue.columns
assert 'action_label' in queue.columns
print('Leakage check passed: no future-window or target-derived columns are in the ranked output.')


Weakest reviewed picks:
 rank  score   reason_code     action_label                                                                           weak_reason
    1     90 stale_visible priority_refresh The score is driven by visibility and weak position more than by a true refresh need.
    2     90 stale_visible priority_refresh The score is driven by visibility and weak position more than by a true refresh need.
    3     90 stale_visible priority_refresh The score is driven by visibility and weak position more than by a true refresh need.
    4     90 stale_visible priority_refresh The score is driven by visibility and weak position more than by a true refresh need.
    5     90 stale_visible priority_refresh The score is driven by visibility and weak position more than by a true refresh need.
    6     90 stale_visible priority_refresh The score is driven by visibility and weak position more than by a true refresh need.
    7     90 stale_visible priority_refresh The score is driven by

## 4) Weak picks + leakage check

The weakest rows are the ones where the rule is still directionally useful but not strong enough to be a confident refresh call. I explicitly confirm there are no future-window or label-derived inputs in the ranked output.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.